# Anomaly detection on images with PCA reconstruction models

Image anomaly detection is aimed at identifying abnormal images in a dataset with a majority of normal images.
The domain is called "Image Anomaly Detection" or "Visual Anomaly Detection".

In this notebook we demonstrate IAD using the public dataset Fashion-MNIST created by Zalando. The dataset is originally created for image classification task. It thus includes 10 classes of images of clothing objects. To use it for an AD task we work in two setups:
1. One vs. One (OVO): one class is defined as the normal (target) class and another class (the abnormal class) is used to seed anomalies into a dataset with a majority of images from the target class.  
2. One vs. all (OVA): one object class is defined as the normal (target) class and all other 9 classes are used to seed anomalies in a dataset composed of a majority of images from the target class. We can also define the One vs Some (OVS) scenario, with anomalied drawn out of several classes.





# 1. DATASET - Fashion MNIST
Fashion-MNIST is a dataset of Zalando's article images—consisting of a training set of 60k examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes



<img src="https://github.com/buehlpa/Anomaly_Detection_Tutorial/blob/main/figures/fmnist_samples_5.png?raw=true" width="800" />

More information: check out the original repo @  https://github.com/zalandoresearch/fashion-mnist

In [ ]:
# @title FMNIST in TSNE 2d Representation
import folium
from PIL import Image
import requests
from io import BytesIO

# Load the image from the URL
url = 'https://github.com/buehlpa/Anomaly_Detection_Tutorial/blob/main/figures/tsne_images_features_all.png?raw=true'  # Replace with your image URL
response = requests.get(url)
img = Image.open(BytesIO(response.content))

# Create a folium map with a larger display size and no tiles
m = folium.Map(location=[0, 0], zoom_start=2, width='70%', height='800px', tiles=None)

# Set the background color to black using custom CSS
m.get_root().html.add_child(folium.Element('''<style>.leaflet-container {background: black;}</style>'''))

# Overlay the image on the map
folium.raster_layers.ImageOverlay(
    image=url,
    bounds=[[-90, -180], [90, 180]],
    opacity=1,
).add_to(m)

# Display the map with the image overlay
m


## Think

Above is the 2D representation of the trainingset of F-MNIST, every point has the according original image attached for a visual interpretation.


- Zoom in and explore the representation , can you spot the different clusters?
- Which anomaly detection tasks (OVA, OVS and OVO) would you expect to be easier? harder?

############################################
# Class names
############################################

class_names

    0. "T-Shirt/Top",    # 0
    1. "Trouser",        # 1
    2. "Pullover",       # 2
    3. "Dress",          # 3
    4. "Coat",           # 4
    5. "Sandal",         # 5
    6. "Shirt",          # 6
    7. "Sneaker",        # 7
    8. "Bag",            # 8
    9. "Ankle Boot"      # 9


# PCA on the raw pixels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from tensorflow.keras.datasets import fashion_mnist
from google.colab import files

## Train PCA on the normal class

In [ ]:
# Select a specific article class to consider as "normal" (e.g., class 0: T-shirts/tops, class 1: trousers)
selected_class = 1
other_classes = [c for c in range(10) if c != selected_class]
# anomaly ratio in the test set
anomaly_ratio = 0.3
# Number of principal components to keep in the PCA reconstruction model
n_components = 130

# Do we want to accelerate the feature extraction by reducing the train and test set sizes?
reduce_size = True


# Load Fashion MNIST data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

if reduce_size:
  # Sample the data to reduce size
  train_indices = np.random.choice(len(x_train), len(x_train) //10 , replace=False)
  test_indices = np.random.choice(len(x_test), len(x_test) // 1, replace=False)

  x_train = x_train[train_indices]
  y_train = y_train[train_indices]
  x_test = x_test[test_indices]
  y_test = y_test[test_indices]


print('trainset size:',len(y_train), ', clean testset size:',len(y_test))
# Normalize the pixel values to the range [0, 1]
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape the images to vectors
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

# train on the selected class
x_train_class = x_train_flat[y_train == selected_class]
# create an anomaly-free test set for visualisation purposes
x_test_class = x_test_flat[y_test == selected_class]

# Fit PCA model on the selected class
pca = PCA(n_components=n_components)
pca.fit(x_train_class)

# Transform the data to the PCA space and then back to reconstruct
x_train_pca = pca.transform(x_train_class)
x_train_reconstructed = pca.inverse_transform(x_train_pca)

x_test_pca = pca.transform(x_test_class)
x_test_reconstructed = pca.inverse_transform(x_test_pca)

# Calculate reconstruction error
train_mse = mean_squared_error(x_train_class, x_train_reconstructed)
test_mse = mean_squared_error(x_test_class, x_test_reconstructed)
print(f"Training MSE: {train_mse}")
print(f"Testing MSE: {test_mse}")

# Visualize original vs reconstructed images
n_images = 5
plt.figure(figsize=(10, 4))

for i in range(n_images):
    # Original image
    plt.subplot(2, n_images, i + 1)
    plt.imshow(x_test_class[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Original")

    # Reconstructed image
    plt.subplot(2, n_images, i + 1 + n_images)
    plt.imshow(x_test_reconstructed[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Reconstructed")

plt.tight_layout()
plt.show()


## Test on a testset with anomalies

In [ ]:
# Create a test set with X% anomalies given by "anomaly_ratio"
n_anomalies = int(anomaly_ratio * len(x_test_class))

anomalous_indices = np.hstack([np.where(y_test == c)[0] for c in other_classes])
anomalous_samples = x_test_flat[np.random.choice(anomalous_indices, n_anomalies, replace=False)]

test_set_with_anomalies = np.vstack((x_test_class, anomalous_samples))
test_labels_with_anomalies = np.hstack((np.zeros(len(x_test_class)), np.ones(n_anomalies)))  # 0 for normal, 1 for anomaly

print('Mixed testset size:',len(test_labels_with_anomalies))

# Reconstruct the test set with anomalies
test_set_reconstructed = pca.inverse_transform(pca.transform(test_set_with_anomalies))

# Calculate reconstruction error for the test set with anomalies
test_mse_with_anomalies = mean_squared_error(test_set_with_anomalies, test_set_reconstructed)
print(f"Test MSE with anomalies: {test_mse_with_anomalies}")

# Visualize some normal and anomalous samples vs reconstructed images
n_images = 5
fig = plt.figure(figsize=(10, 8))

for i in range(n_images):
    # Original normal image
    plt.subplot(4, n_images, i + 1)
    plt.imshow(x_test_class[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Normal Input")

    # Reconstructed normal image
    plt.subplot(4, n_images, i + 1 + n_images)
    plt.imshow(x_test_reconstructed[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Normal Recon.")

for i in range(n_images):
    # Original anomalous image
    plt.subplot(4, n_images, i + 1 + 2 * n_images)
    plt.imshow(anomalous_samples[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Anomalous Input")

    # Reconstructed anomalous image
    plt.subplot(4, n_images, i + 1 + 3 * n_images)
    plt.imshow(test_set_reconstructed[len(x_test_class) + i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Anomalous Recon.")

plt.tight_layout()
plt.show()

#fig.savefig('fmnist_images_recon.pdf')  # Save the figure

# Download the figure
#from google.colab import files
#files.download('fmnist_images_recon.pdf')

In [ ]:
test_set_with_anomalies.shape

## Histogram of reconstruction errors

In [ ]:
# Calculate MSE for individual samples
train_mse_values = np.mean((x_train_class - x_train_reconstructed) ** 2, axis=1)
test_mse_values = np.mean((test_set_with_anomalies - test_set_reconstructed) ** 2, axis=1)

test_mse_anomalies = test_mse_values[test_labels_with_anomalies==1]
test_mse_normal = test_mse_values[test_labels_with_anomalies==0]

In [ ]:

# Plot histograms of MSE values
fig = plt.figure(figsize=(4, 3))
plt.hist(train_mse_values, bins=50, density=True,alpha=0.5, label="Train MSE")
plt.hist(test_mse_normal, bins=50, density=True, alpha=0.5, label="Test MSE normal")
plt.hist(test_mse_anomalies, bins=50, density=True, alpha=0.5, label="Test MSE anomalies")
plt.xlabel("Mean Squared Error")
plt.ylabel("Frequency")
plt.ylim(0,60)
plt.legend()
plt.title("Reconstruction Errors PCA on Pixels")

plt.tight_layout()
plt.show()
#fig.savefig('fmnist_hist_PCA_pixels.pdf')  # Save the figure

# Download the figure
#from google.colab import files
#files.download('fmnist_hist_PCA_pixels.pdf')

## Evaluation: precision recall curve

In [ ]:
# Apply Min-Max Normalization to the test scores
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import norm

scaler = MinMaxScaler()
normalized_scores = scaler.fit_transform(test_mse_values.reshape(-1, 1)).flatten()


In [ ]:
anomaly_scores = normalized_scores
true_labels = test_labels_with_anomalies

### Function to calculate precision and recall

In [ ]:
def calculate_precision_recall_from_scratch(anomaly_scores, true_labels, threshold_array):
    """
    Calculate precision and recall for a given set of thresholds from scratch.

    Parameters:
    - anomaly_scores: Array-like, the anomaly scores output by the isolation forest.
                      Higher scores indicate a higher likelihood of being an anomaly.
    - true_labels: Array-like, ground truth labels (0 for normal, 1 for anomalies).
    - threshold_array: List or array of thresholds to evaluate.

    Returns:
    - precision: List of precision values for each threshold.
    - recall: List of recall values for each threshold.
    """
    precision = []
    recall = []

    for threshold in threshold_array:
        # Classify points as anomalies (1) or normal (0) based on the threshold
        predicted_labels = (anomaly_scores >= threshold).astype(int)

        # True Positives, False Positives, False Negatives
        tp = ((predicted_labels == 1) & (true_labels == 1)).sum()
        fp = ((predicted_labels == 1) & (true_labels == 0)).sum()
        fn = ((predicted_labels == 0) & (true_labels == 1)).sum()

        # Precision and Recall calculations
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0

        precision.append(prec)
        recall.append(rec)

    return precision, recall

### define threshold values

In [ ]:
threshold_arr = np.arange(0,1,0.01)
precision_test, recall_test = calculate_precision_recall_from_scratch(anomaly_scores, true_labels, threshold_arr)

### plot precision-recall

In [ ]:
plt.figure(figsize=(4, 4))
plt.plot(recall_test,precision_test,'ko-',ms=3)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid()

plt.tight_layout()
plt.tight_layout()
#plt.savefig('prc_PCA_pixels_class1vsall_130PCs.pdf')  # Save the figure
plt.show()

# Download the figure
#files.download('prc_PCA_pixels_class1vsall_130PCs.pdf')

## Think
1. Explore the effect of the number of principal components on the outcome. How would you select this parameter?
2. Can you extend the evaluation with a ROC plot?
3. Can you explain the performance? Hint: which is the normal class and which are the anomalies?
4. Can you design an "easier" AD task with the FMNIST data? a harder one?


# Autoencoder (AE) on raw pixels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.image import resize
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dropout

import os


 ## Train AE model

In [ ]:
# Define autoencoder model
encoding_dim = 64  # Dimensionality of encoded space
input_dim = x_train_class.shape[1]

autoencoder = Sequential([
    Input(shape=(input_dim,)),
    Dense(256, activation='relu'),
    #Dense(128, activation='relu'),
    #Dropout(0.2),
    Dense(encoding_dim, activation='relu'),
    #Dropout(0.2),
    #Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    #Dropout(0.2),
    Dense(input_dim, activation='linear')
])


autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# Train the autoencoder
autoencoder.fit(x_train_class, x_train_class, epochs=50, batch_size=64, shuffle=True, validation_split=0.2)


## Reconstruct using the trained model

In [ ]:
# Reconstruct train and test sets using the autoencoder
x_train_reconstructed_AE = autoencoder.predict(x_train_class)
x_test_reconstructed_AE = autoencoder.predict(test_set_with_anomalies)



## Reconstruction errors

In [ ]:
# Calculate reconstruction error for the test set with anomalies
test_mse_with_anomalies_AE = mean_squared_error(test_set_with_anomalies, x_test_reconstructed_AE)
print(f"Test MSE with anomalies: {test_mse_with_anomalies_AE}")

# Calculate MSE for individual samples
train_mse_values_AE = np.mean((x_train_class - x_train_reconstructed_AE) ** 2, axis=1)
test_mse_values_AE = np.mean((test_set_with_anomalies - x_test_reconstructed_AE) ** 2, axis=1)

test_mse_anomalies_AE = test_mse_values_AE[test_labels_with_anomalies==1]
test_mse_normal_AE = test_mse_values_AE[test_labels_with_anomalies==0]
# Plot histograms of MSE values
fig = plt.figure(figsize=(4, 3))
plt.hist(train_mse_values_AE, bins=50, density=True,alpha=0.5, label="Train MSE")
plt.hist(test_mse_normal_AE, bins=50, density=True, alpha=0.5, label="Test MSE normal")
plt.hist(test_mse_anomalies_AE, bins=50, density=True, alpha=0.5, label="Test MSE anomalies")
plt.xlabel("Mean Squared Error")
plt.ylabel("Frequency")
plt.ylim(0,60)
plt.legend()
plt.title("Reconstruction Errors AE on Pixels")

plt.tight_layout()
plt.show()
#fig.savefig('fmnist_hist_recon_AE.pdf')  # Save the figure

# Download the figure
#from google.colab import files
#files.download('fmnist_hist_recon_AE.pdf')

## Visualize original and reconstructed images

In [ ]:
# Visualize some normal and anomalous samples vs reconstructed images
n_images = 5
fig = plt.figure(figsize=(10, 8))

for i in range(n_images):
    # Original normal image
    plt.subplot(4, n_images, i + 1)
    plt.imshow(x_test_class[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Normal Input")

    # Reconstructed normal image
    plt.subplot(4, n_images, i + 1 + n_images)
    plt.imshow(x_test_reconstructed_AE[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Normal Recon.")

for i in range(n_images):
    # Original anomalous image
    plt.subplot(4, n_images, i + 1 + 2 * n_images)
    plt.imshow(anomalous_samples[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Anomalous Input")

    # Reconstructed anomalous image
    plt.subplot(4, n_images, i + 1 + 3 * n_images)
    plt.imshow(x_test_reconstructed_AE[len(x_test_class) + i].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.title("Anomalous Recon.")

plt.tight_layout()
plt.show()

#fig.savefig('fmnist_images_recon_AE.pdf')  # Save the figure

# Download the figure
#from google.colab import files
#files.download('fmnist_images_recon_AE.pdf')

## Evaluate: precision recall curve

In [ ]:
# Apply Min-Max Normalization to the test scores
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import norm

scaler = MinMaxScaler()
normalized_scores = scaler.fit_transform(test_mse_values_AE.reshape(-1, 1)).flatten()
anomaly_scores_AE = normalized_scores
true_labels_AE = test_labels_with_anomalies

In [ ]:
threshold_arr = np.arange(0,1.0,0.01)
precision_test_AE, recall_test_AE = calculate_precision_recall_from_scratch(anomaly_scores_AE, true_labels_AE, threshold_arr)

In [ ]:
plt.figure(figsize=(4, 4))
plt.plot(recall_test,precision_test,'ko-',ms=3,label='PCA pixels')
plt.plot(recall_test_AE,precision_test_AE,'ro-',ms=3,label='AE pixels')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid()
plt.legend()
plt.tight_layout()
#plt.savefig('fmnist_prc_easy_PCA_AE_256_64_256.pdf')  # Save the figure
plt.show()

# Download the figure
#files.download('fmnist_prc_easy_PCA_AE_256_64_256.pdf')

## Think
1. Play with different scenarios (normal/abnormal classes) and compare the performance of the PCA and AE.
2. What is the challenge in the architecture optimization here? Hint: think of the inherent difference between the validation and test data.
3. Can you come up with a procedure to do it systematically?